# 01 EDA – skan surowych plików

## 1. Konfiguracja

In [1]:
import sys
sys.path.append("..")

import json
import re
from collections import Counter

import pandas as pd
from tqdm.auto import tqdm

from src.config import LINK_MEDIA_PATTERN, PHASES, PROCESSED_DIR, RAW_DIR, STUDY_FILES, WINDOWS
from src.weeks import to_week_start

SCAN_DIR = PROCESSED_DIR / "scan"
SCAN_DIR.mkdir(parents=True, exist_ok=True)
link_media = re.compile(LINK_MEDIA_PATTERN)
MIN_COMMENTS_TO_CHECK = [50, 100, 200, 300, 500, 1000, 2000, 5000]
known_bots = pd.read_csv(PROCESSED_DIR / "known_bots.csv")["author"].str.lower().tolist()

c:\Users\karol\Licencjat_programowanie\Licencjat\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Fazy cyklu

In [2]:
phases = pd.DataFrame([(cycle, phase, start, end) for cycle, rows in PHASES.items() for phase, start, end in rows],
                      columns=["cycle", "phase", "start", "end"])
phases.to_csv(PROCESSED_DIR / "phases.csv", index=False)
phases.groupby("cycle").size()

cycle
2012    5
2016    5
2020    5
2024    5
dtype: int64

## 3. Lista plików

In [3]:
expected = {f"r_{subreddit}_comments_{cycle}.jsonl" for subreddit, cycle in STUDY_FILES}
found = {path.name for path in RAW_DIR.glob("*.jsonl")}
print("oczekiwane:", len(expected), "| brakujące:", sorted(expected - found))
print("nadmiarowe (poza badaniem):", sorted(found - expected))

oczekiwane: 30 | brakujące: []
nadmiarowe (poza badaniem): []


## 4. Skan plików

In [4]:
file_rows, weekly_tables, schema_rows, percentile_rows, threshold_rows, top_rows, known_rows = [], [], [], [], [], [], []

for subreddit, cycle in tqdm(STUDY_FILES):
    path = RAW_DIR / f"r_{subreddit}_comments_{cycle}.jsonl"
    window_start, window_end = WINDOWS[cycle]
    start_ts = pd.Timestamp(window_start).timestamp()
    end_ts = pd.Timestamp(window_end).timestamp() + 86400

    field_counts = Counter()
    rows = []
    n_bad_lines = 0
    with open(path, encoding="utf-8") as file:
        for line in file:
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                n_bad_lines += 1
                continue
            field_counts.update(record.keys())
            body = record.get("body") or ""
            rows.append((int(record["created_utc"]), record.get("retrieved_on"), record.get("author"), len(body.split()),
                         body.strip() in ("[deleted]", "[removed]"), record.get("distinguished") == "moderator",
                         bool((record.get("_meta") or {}).get("removal_type")), bool(link_media.search(body))))
    df = pd.DataFrame(rows, columns=["created_utc", "retrieved_on", "author", "n_words", "deleted", "moderator",
                                     "removed_later", "has_link"])
    del rows
    n_records = len(df)

    for field, count in field_counts.items():
        schema_rows.append({"subreddit": subreddit, "cycle": cycle, "field": field, "share": count / n_records})

    inside = (df["created_utc"] >= start_ts) & (df["created_utc"] < end_ts)
    delay_days = ((df["retrieved_on"] - df["created_utc"]) / 86400)[inside]
    file_rows.append({
        "subreddit": subreddit, "cycle": cycle, "size_mb": round(path.stat().st_size / 1e6, 1), "n_records": n_records,
        "n_bad_lines": n_bad_lines, "n_outside_window": int((~inside).sum()),
        "first_utc": pd.to_datetime(df["created_utc"].min(), unit="s"),
        "last_utc": pd.to_datetime(df["created_utc"].max(), unit="s"),
        "retrieved_share": delay_days.notna().mean(), "delay_median_days": delay_days.median(),
        "delay_p90_days": delay_days.quantile(0.9),
    })

    df = df[inside].copy()
    df["week_start"] = to_week_start(df["created_utc"])
    content = df[~df["deleted"]]
    by_week = df.groupby("week_start")
    by_week_content = content.groupby("week_start")
    weekly = pd.DataFrame({"n_raw": by_week.size(), "n_deleted": by_week["deleted"].sum(),
                           "n_moderator": by_week["moderator"].sum(), "n_removed_later": by_week["removed_later"].sum(),
                           "n_content": by_week_content.size(), "n_link": by_week_content["has_link"].sum()})
    for words in (3, 5, 10):
        weekly[f"n_ge_{words}w"] = content[content["n_words"] >= words].groupby("week_start").size()
    weekly["n_authors"] = content[content["author"] != "[deleted]"].groupby("week_start")["author"].nunique()
    weekly = weekly.reindex(pd.date_range(window_start, window_end, freq="7D").astype("datetime64[s]")).fillna(0).astype(int)
    weekly.index.name = "week_start"
    weekly.insert(0, "subreddit", subreddit)
    weekly.insert(1, "cycle", cycle)
    weekly_tables.append(weekly.reset_index())
    file_rows[-1]["n_empty_weeks"] = int((weekly["n_raw"] == 0).sum())

    author_counts = content.loc[content["author"] != "[deleted]", "author"].value_counts()
    n_comments = len(content)
    percentile_rows.append({"subreddit": subreddit, "cycle": cycle, "n_authors": len(author_counts), "n_comments": n_comments,
                            "p50": author_counts.quantile(0.5), "p90": author_counts.quantile(0.9),
                            "p99": author_counts.quantile(0.99), "p999": author_counts.quantile(0.999),
                            "max": author_counts.max()})
    for min_comments in MIN_COMMENTS_TO_CHECK:
        high = author_counts[author_counts >= min_comments]
        threshold_rows.append({"subreddit": subreddit, "cycle": cycle, "min_comments": min_comments,
                               "n_accounts": len(high), "n_comments": int(high.sum()),
                               "share_comments": high.sum() / n_comments})
    for author, n in author_counts.head(20).items():
        top_rows.append({"kind": "top_account", "subreddit": subreddit, "cycle": cycle, "author": author, "n": n,
                         "share": n / n_comments})
    counts_by_lowercase = author_counts.groupby(author_counts.index.str.lower()).sum()
    for bot in known_bots:
        known_rows.append({"subreddit": subreddit, "cycle": cycle, "author": bot, "n": int(counts_by_lowercase.get(bot, 0))})
    del df, content

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [24:01<00:00, 48.05s/it] 


## 5. Zapis tabel

In [ ]:
pd.DataFrame(file_rows).to_csv(SCAN_DIR / "files.csv", index=False)
pd.concat(weekly_tables).to_csv(SCAN_DIR / "weekly.csv", index=False)
pd.DataFrame(schema_rows).to_csv(SCAN_DIR / "schema.csv", index=False)
pd.DataFrame(percentile_rows).to_csv(SCAN_DIR / "author_percentiles.csv", index=False)
pd.DataFrame(threshold_rows).to_csv(SCAN_DIR / "author_thresholds.csv", index=False)
pd.DataFrame(top_rows).to_csv(PROCESSED_DIR / "bot_candidates.csv", index=False)
pd.DataFrame(known_rows).to_csv(PROCESSED_DIR / "known_bot_counts.csv", index=False)
pd.DataFrame(file_rows)[["subreddit", "cycle", "size_mb", "n_records", "n_bad_lines", "n_outside_window", "n_empty_weeks"]]

,subreddit,cycle,size_mb,n_records,n_bad_lines,n_outside_window,n_empty_weeks
0,democrats,2012,10.9,15002,0,0,0
1,democrats,2016,73.7,109049,0,0,0
2,democrats,2020,455.9,351984,0,0,0
3,democrats,2024,1999.6,1171188,0,0,0
4,Liberal,2012,20.7,25914,0,0,0
5,Liberal,2016,44.2,58705,0,0,0
6,Liberal,2020,98.8,76403,0,0,0
7,Liberal,2024,291.7,156937,0,0,0
8,Republican,2012,78.0,101372,0,0,0
9,Republican,2016,99.5,143316,0,0,0


: 